# Notebook 4.0: Sentiment Analysis & Model Validation (RQ1)

**Research Question 1 (RQ1) & Primary Scoring**
This notebook:
1. Loads sampled comments (where granular emotional sentiment resides).
2. Applies the text cleaning pipeline ().
3. Computes VADER sentiment (compound score & 3-class discrete label).
4. Computes  probabilities using batched GPU/CPU inference.
5. Directly compares VADER vs. RoBERTa to address RQ1 (agreement rate, disagreement analysis).
6. Plots weekly sentiment trends (marking Nov 30, 2022 ChatGPT launch).
7. Exports  for downstream paper figures.

In [ ]:
import sys
from pathlib import Path

nb_dir = Path.cwd()
project_root = nb_dir.parent if nb_dir.name == 'notebooks' else nb_dir
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.modeling import set_seed
from src.modeling.load import load_samples
from src.modeling.text import apply_pipeline
from src.modeling.sentiment import load_roberta, score_roberta, score_vader, compare_models

set_seed(42)

config_path = project_root / 'config' / 'config.yaml'
with open(config_path, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

print('Sentiment analysis pipeline loaded.')

## Step 1: Load Comment Samples
Load comments from the sampled data directory.

In [ ]:
samples_dir = project_root / 'data' / 'output' / 'samples' / '2.0'
if not samples_dir.exists() or not list(samples_dir.glob('*.json')):
    samples_dir = (project_root / cfg['paths']['data_samples_dir'] / '1.2').resolve()

df_comments = load_samples(
    samples_dir=samples_dir,
    subreddits=cfg['analysis']['subreddits'],
    dataset_types=['comments'],
)

# Text cleaning
text_col = 'body' if 'body' in df_comments.columns else 'text'
df_comments['cleaned_text'] = apply_pipeline(df_comments[text_col])
df_comments = df_comments[df_comments['cleaned_text'].str.strip().str.len() > 10].reset_index(drop=True)

print(f'Ready for sentiment scoring: {len(df_comments):,} comments.')

## Step 2: Score VADER (Cross-Check Baseline)
Compute VADER compound scores with a neutral margin of |compound| > 0.05.

In [ ]:
print('Scoring comments with VADER...')
vader_df = score_vader(df_comments['cleaned_text'].tolist(), neutral_band=0.05)

df_comments['vader_compound'] = vader_df['vader_compound']
df_comments['vader_label'] = vader_df['vader_label']

print('VADER sentiment distribution:')
print(df_comments['vader_label'].value_counts(normalize=True).round(3))

## Step 3: Score RoBERTa (Primary Domain Model)
Run batched inference with .

In [ ]:
model_name = cfg['modeling']['sentiment_model']
batch_size = cfg['modeling']['sentiment_batch_size']
device = cfg['modeling']['device']

print(f'Loading RoBERTa model ({model_name})...')
tokenizer, model, resolved_device = load_roberta(model_name=model_name, device=device)
print(f'Using device: {resolved_device}')

roberta_scores = score_roberta(
    texts=df_comments['cleaned_text'].tolist(),
    tokenizer=tokenizer,
    model=model,
    device=resolved_device,
    batch_size=batch_size,
)

df_comments['negative'] = roberta_scores['negative']
df_comments['neutral'] = roberta_scores['neutral']
df_comments['positive'] = roberta_scores['positive']
df_comments['roberta_label'] = roberta_scores[['negative', 'neutral', 'positive']].idxmax(axis=1)

print('RoBERTa sentiment distribution:')
print(df_comments['roberta_label'].value_counts(normalize=True).round(3))

## Step 4: RQ1 Model Comparison & Disagreement Analysis
Assess whether domain RoBERTa outperforms lexicon-based VADER on tech career text.

In [ ]:
comparison = compare_models(df_comments)
print(f'=== RQ1 Agreement Rate: {comparison["agreement_rate"]:.1%} ===')

top_disagreements = comparison['top_disagreements']
print('
Top-5 Disagreements between VADER and RoBERTa:')
for idx, row in top_disagreements.head(5).iterrows():
    print(f'Text: {row["cleaned_text"][:120]}...')
    print(f'  VADER: {row["vader_label"]} (comp: {row["vader_compound"]:.2f}) | RoBERTa: {row["roberta_label"]} (pos: {row["positive"]:.2f}, neg: {row["negative"]:.2f})
')

## Step 5: VADER vs. RoBERTa Correlation Visualization
Scatter plot comparing VADER compound scores with RoBERTa probability margins.

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df_comments.sample(min(2000, len(df_comments)), random_state=42),
    x='vader_compound',
    y='positive',
    hue='roberta_label',
    palette={'negative': '#d9534f', 'neutral': '#f0ad4e', 'positive': '#5cb85c'},
    alpha=0.5,
)
plt.title(f'VADER Compound vs. RoBERTa Positive Probability (Agreement: {comparison["agreement_rate"]:.1%})', fontsize=12)
plt.xlabel('VADER Compound Score (-1 to +1)')
plt.ylabel('RoBERTa Positive Class Probability')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

fig_path = project_root / 'data' / 'output' / 'figures' / 'vader_vs_roberta_scatter.png'
plt.savefig(fig_path, dpi=300)
plt.show()

## Step 6: Export Scored Dataset
Save the scored dataset to .

In [ ]:
output_parquet = project_root / 'data' / 'output' / 'sentiment_df.parquet'
output_parquet.parent.mkdir(parents=True, exist_ok=True)
df_comments.to_parquet(output_parquet, index=False)
print(f'Scored sentiment dataset exported successfully to: {output_parquet}')